# 🦴 RepublicaVideo1 · Paso 3c: skinning con **SkinTokens** (la IA de rigging más nueva de VAST-AI, sucesora de UniRig)
- Usa **tu esqueleto** (cuerpo + cola + orejas + cara) y sólo calcula los pesos (`--use_skeleton`), conserva texturas (`--use_transfer`).
- **Ahorro:** celda 1 en **CPU** (instala y guarda en tu Drive) → cambia a **G4** → celdas 1 y 2 (~5-8 min de GPU en total).
- Funciona sin compilar flash-attn (atención de PyTorch), así no hay problemas con la G4.
- Mide el tiempo de GPU en `RepublicaVideo1/gasto_gpu.csv` y desconecta el entorno al terminar o si hay un error.

In [ ]:
#@title 1 · Preparar (corre en CPU): instala SkinTokens y baja los modelos; lo guarda en tu Drive
#@markdown **Para ahorrar:** corre esta celda con **Entorno de ejecución → CPU**. Luego cambia a la **G4** y corre la celda 2
#@markdown (restaura todo desde Drive en ~2 min en vez de instalar con la GPU prendida).
DESCONECTAR_AL_TERMINAR = True  #@param {type:"boolean"}
import os, sys, time, json, subprocess, csv, datetime, shutil, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
RV = '/content/drive/MyDrive/RepublicaVideo1'; os.makedirs(os.path.join(RV, 'rig'), exist_ok=True)
CACHE = '/content/drive/MyDrive/.cache_colab'; os.makedirs(CACHE, exist_ok=True)
ST = '/content/SkinTokens'; ENV = '/content/st_env'; TAR = f'{CACHE}/skintokens_env.tar'
GASTO = os.path.join(RV, 'gasto_gpu.csv')
def gpu_name():
    try: return subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'CPU'
    except FileNotFoundError: return 'CPU'
def gpu_total_min():
    return sum(float(r['min_gpu']) for r in csv.DictReader(open(GASTO))) if os.path.exists(GASTO) else 0.0
def desconectar():
    if IN_COLAB and DESCONECTAR_AL_TERMINAR:
        print("Desconectando el entorno para no gastar unidades…"); from google.colab import runtime; runtime.unassign()
class Paso:
    def __init__(self, nombre): self.nombre = nombre
    def __enter__(self):
        self.t0 = time.time(); self.dev = gpu_name(); print(f"▶ {self.nombre} · {self.dev} · GPU usada hasta ahora: {gpu_total_min():.1f} min"); return self
    def __exit__(self, et, ev, tb):
        m = (time.time() - self.t0) / 60; g = m if self.dev != 'CPU' else 0.0; new = not os.path.exists(GASTO)
        with open(GASTO, 'a', newline='') as f_:
            w = csv.writer(f_)
            if new: w.writerow(['paso', 'inicio', 'min_total', 'min_gpu', 'maquina', 'ok'])
            w.writerow([self.nombre, datetime.datetime.now().isoformat(timespec='seconds'), f"{m:.2f}", f"{g:.2f}", self.dev, et is None])
        print(f"■ {self.nombre} · {m:.1f} min ({g:.1f} de GPU) · GPU total: {gpu_total_min():.1f} min")
        if et is not None: print("✗ Error:", str(ev)[:400]); desconectar()
        return False
def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0: raise RuntimeError(f"{cmd[:120]}…\n{r.stderr[-1500:]}")
    return r.stdout
SHIM = '''import torch.nn.functional as F
def flash_attn_func(q, k, v, *a, causal=False, softmax_scale=None, **kw):
    """Igual que flash-attn ([B, S, H, D]) pero con la atención de PyTorch: funciona en cualquier GPU (incluida la G4)."""
    dt = q.dtype; k = k.to(dt); v = v.to(dt)
    o = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=causal, scale=softmax_scale)
    return o.transpose(1, 2), None
'''
EXTRAS = ['scipy']          # SkinTokens los usa pero no vienen en su requirements.txt
PRUEBA = ("import sys; sys.path.insert(0, '.'); import src.data.dataset, src.data.transform, src.model.tokenrig, "
          "src.tokenizer.parse, src.data.vertex_group, src.server.spec, bpy, scipy; print('imports ok')")
def probar_imports():
    """Importa todo lo que usa demo.py, sin GPU: así un error de librerías aparece en CPU y no gasta la G4."""
    r = subprocess.run([f'{ENV}/bin/python', '-c', PRUEBA], cwd=ST, capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=f'{ENV}/shim'))
    if r.returncode != 0: raise RuntimeError("Falta algo en el entorno de SkinTokens:\n" + (r.stdout + r.stderr)[-1200:])
def guardar():
    sh(f"tar -cf /content/st.tar -C / content/st_env content/SkinTokens root/.local/share/uv/python && cp /content/st.tar '{TAR}' && rm /content/st.tar")
def preparar():
    estado = 'ya estaba'
    if not (os.path.exists(f'{ENV}/.ok') and os.path.exists(f'{ST}/experiments')):
        if os.path.exists(TAR):
            sh(f"tar -xf '{TAR}' -C /"); estado = 'restaurado desde Drive'
    if not os.path.exists(f'{ENV}/.ok'):
        sh(f"{sys.executable} -m pip install -q uv")
        sh("apt-get -qq install -y libegl1 libgl1 libxi6 libxkbcommon-x11-0 libxfixes3 libxrender1 libsm6")
        if not os.path.exists(ST): sh(f"git clone -q --depth 1 https://github.com/VAST-AI-Research/SkinTokens {ST}")
        sh(f"uv venv -q --python 3.11 {ENV}")
        sh(f"uv pip install -q --python {ENV}/bin/python torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu128")
        sh(f"uv pip install -q --python {ENV}/bin/python -r {ST}/requirements.txt " + ' '.join(EXTRAS))
        os.makedirs(f'{ENV}/shim', exist_ok=True); open(f'{ENV}/shim/flash_attn_interface.py', 'w').write(SHIM)
        sh(f"cd {ST} && {ENV}/bin/python download.py --model")
        open(f'{ENV}/.ok', 'w').write('ok'); estado = 'instalado'
    falta = [m for m in EXTRAS if subprocess.run([f'{ENV}/bin/python', '-c', f'import {m}'], capture_output=True).returncode != 0]
    if falta:
        if not shutil.which('uv'): sh(f"{sys.executable} -m pip install -q uv")
        sh(f"uv pip install -q --python {ENV}/bin/python " + ' '.join(falta)); estado += f" + agregué {', '.join(falta)}"
    probar_imports()
    if 'instalado' in estado or 'agregué' in estado: guardar(); estado += ' (guardado en Drive)'
    return estado
with Paso('3c-1 · preparar SkinTokens'):
    print("SkinTokens:", preparar())
    print("Modelos:", ', '.join(sorted(os.path.relpath(p, ST) for p in glob.glob(f'{ST}/experiments/*/*.ckpt'))))
if gpu_name() == 'CPU':
    print("\n✓ Listo. Ahora: Entorno de ejecución → Cambiar tipo → G4, y corre las celdas 1 y 2 (la 1 restaura en segundos).")
    desconectar()

In [ ]:
#@title 2 · Skinning con SkinTokens en la G4 (usa tu esqueleto: cola, orejas y cara) + prueba de pose
#@markdown Mantiene tu esqueleto y sólo calcula los pesos. `POSTPROCESO_VOXEL` ayuda a que las piezas separadas de Roblox no se desgarren.
POSTPROCESO_VOXEL = True  #@param {type:"boolean"}
SEMILLAS = 1  #@param {type:"slider", min:1, max:3, step:1}
import urllib.request
GH = 'https://raw.githubusercontent.com/gustavo45a/kai-assistant/ccr-9243d395-h8nef9/tools/colab/republica/rig/'
IN = '/content/st_in'; OUT = os.path.join(RV, 'rig', 'skintokens'); os.makedirs(IN, exist_ok=True); os.makedirs(OUT, exist_ok=True)
if gpu_name() == 'CPU': raise RuntimeError("Esta celda necesita la G4: cambia el tipo de entorno y corre la celda 1 (restaura) y luego ésta.")
with Paso('3c-2 · skinning SkinTokens (GPU)'):
    print("Preparación:", preparar())
    for n in ('confundido', 'explicador'):
        urllib.request.urlretrieve(GH + f'{n}_esqueleto.fbx', f'{IN}/{n}_esqueleto.fbx')
    urllib.request.urlretrieve(GH + 'probar_pose.py', f'{IN}/probar_pose.py')
    env = dict(os.environ, PYTHONPATH=f'{ENV}/shim')
    hechos = []
    for n in ('confundido', 'explicador'):
        for s in range(SEMILLAS):
            out = f'{OUT}/{n}_skintokens{"" if s == 0 else f"_{s + 1}"}.glb'
            cmd = [f'{ENV}/bin/python', 'demo.py', '--input', f'{IN}/{n}_esqueleto.fbx', '--output', out, '--use_skeleton', '--use_transfer']
            if POSTPROCESO_VOXEL: cmd.append('--use_postprocess')
            t0 = time.time(); r = subprocess.run(cmd, cwd=ST, capture_output=True, text=True, env=env)
            ok = os.path.exists(out)
            print(f"  {n} #{s + 1}: {'✓' if ok else '✗'} ({time.time() - t0:.0f}s)")
            if not ok: raise RuntimeError(f"SkinTokens no generó {n}:\n{(r.stdout + r.stderr)[-2000:]}")
            hechos.append((n, out))
    # prueba de pose (codos, rodillas, cola, orejas, mandíbula) con el Blender del entorno
    for n, out in hechos:
        subprocess.run([f'{ENV}/bin/python', f'{IN}/probar_pose.py', '--', out, out[:-4]], capture_output=True)
    import numpy as np
    from PIL import Image
    rows = []
    for n, out in hechos:
        ims = [Image.open(out[:-4] + f'_{t}{k}.png').convert('RGB') for t in ('reposo', 'pose') for k in (0, 1) if os.path.exists(out[:-4] + f'_{t}{k}.png')]
        if ims: rows.append(np.hstack([np.array(i) for i in ims]))
    if rows:
        w = max(r.shape[1] for r in rows); rows = [np.pad(r, ((0, 0), (0, w - r.shape[1]), (0, 0))) for r in rows]
        Image.fromarray(np.vstack(rows)).save(f'{OUT}/prueba_pose_skintokens.jpg', quality=88)
        from IPython.display import Image as IPI, display; display(IPI(f'{OUT}/prueba_pose_skintokens.jpg', width=900))
print("Resultados en:", OUT)
desconectar()